In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import gc

@dataclass
class SequenceStore:
    # Per-row arrays
    species_ids: np.ndarray
    gene_ids: np.ndarray
    sequences: list[np.ndarray]
    # Lookup tables
    species_to_gene_to_rows: dict[int, dict[int, list[int]]]
    valid_species: np.ndarray
    # Name lookups
    species_names: list[str]
    gene_names: list[str]
    # Token map used for pre-encoding
    token_to_id: dict[str, int]

    @classmethod
    def from_parquet(cls, parquet_path: str | Path, order_col: str = "order", family_col: str = "family",
                     genus_col: str = "genus", species_col: str = "species", gene_col: str = "gene", 
                     sequence_col: str = "sequence", select_orders: list[str] | None = None,
                     select_families: list[str] | None = None, select_genera: list[str] | None = None,
                     select_species: list[str] | None = None, select_genes: list[str] | None = None) -> "SequenceStore":
        print("[SequenceStore] - Loading Parquet file")
        df = pd.read_parquet(parquet_path, columns=[order_col, family_col, genus_col, species_col, gene_col,
                                                    sequence_col])

        # Filter by taxonomy or genes if provided
        for select_list, select_col in [
            (select_orders, order_col), (select_families, family_col),(select_genera, genus_col),
            (select_species, species_col), (select_genes, gene_col)]:
            if select_list is not None:
                print(f"[SequenceStore] - Filtering column {select_col} for {select_list}")
                df = df[df[select_col].isin(select_list)]
                
                # Optional: Check if any rows remain
                if df.empty:
                    raise ValueError("No rows remain after filtering")

        # Basic checks
        print("[SequenceStore] - Checking for missing values")
        if df[species_col].isna().any():
            raise ValueError(f"Column '{species_col}' contains missing values.")
        if df[gene_col].isna().any():
            raise ValueError(f"Column '{gene_col}' contains missing values.")
        if df[sequence_col].isna().any():
            raise ValueError(f"Column '{sequence_col}' contains missing values.")
        
        # Ensure categorical dtype, and remove unused categories to ensure continuous numbering
        print("[SequenceStore] - Ensuring continuous categorical species and gene columns")
        species_cat = df[species_col].astype('category').cat.remove_unused_categories()
        gene_cat = df[gene_col].astype('category').cat.remove_unused_categories()

        # Extract category integer IDs
        print("[SequenceStore] - Extracting category integer IDs")
        species_ids = species_cat.cat.codes.to_numpy(copy=True)
        gene_ids = gene_cat.cat.codes.to_numpy(copy=True)

        # Extract category names
        print("[SequenceStore] - Extracting category names")
        species_names = species_cat.cat.categories.to_list()
        gene_names = gene_cat.cat.categories.to_list()

        #Pre-encode sequences, using tiny vocabulary for DNA / ambiguous bases
        print("[SequenceStore] - Pre-encoding sequences, using tiny vocabulary for DNA / ambiguous bases ")
        token_to_id = {
            'PAD': 0,
            'A': 1,
            'C': 2,
            'G': 3,
            'T': 4,
            'N': 5,
        }

        encode_array = np.full(256, token_to_id['N'], dtype=np.uint8)
        for token, idx in token_to_id.items():
            if token != 'PAD':
                encode_array[ord(token)] = idx

        def encode_dna(seq):
        # Convert string to ASCII bytes, then use the encode_array as a lookup table
            return encode_array[np.frombuffer(seq.encode('ascii'), dtype=np.uint8)]
        
        sequences = [encode_dna(seq) for seq in df[sequence_col].array]
        
        # Build nested row-index mapping
        print("[SequenceStore] - Building nested row-index mapping")
        species_to_gene_to_rows = defaultdict(lambda: defaultdict(list))
        for row_idx, (sid, gid) in enumerate(zip(species_ids, gene_ids)):
            species_to_gene_to_rows[int(sid)][int(gid)].append(row_idx)

        species_to_gene_to_rows = {
            sid: dict(gene_map)
            for sid, gene_map in species_to_gene_to_rows.items()
        }

        valid_species = np.array(sorted(species_to_gene_to_rows.keys()))

        print("[SequenceStore] - Cleaning up temporary memory")
        del df, species_cat, gene_cat
        gc.collect()

        return cls(
            species_ids=species_ids,
            gene_ids=gene_ids,
            sequences=sequences,
            species_to_gene_to_rows=species_to_gene_to_rows,
            valid_species=valid_species,
            species_names=species_names,
            gene_names=gene_names,
            token_to_id=token_to_id
        )
    
    @property
    def num_rows(self) -> int:
        return len(self.sequences)
    
    @property
    def num_species(self) -> int:
        return len(self.species_names)
    
    @property
    def num_valid_species(self) -> int:
        return len(self.valid_species)
    
    @property
    def num_genes(self) -> int:
        return len(self.gene_names)
    
    @property
    def vocab_size(self) -> int:
        return len(self.token_to_id)
    
    def summary(self) -> str:
        return (
            f"SequenceStore(num_rows={self.num_rows}, "
            f"num_species={self.num_species}, "
            f"num_genes={self.num_genes}, "
            f"valid_species={self.num_valid_species})"
        )

In [ ]:
#select_orders = ['Cypriniformes', 'Perciformes', 'Siluriformes', 'Gobiiformes']
select_orders = None
select_families = None
select_genera = None
select_species = None
select_genes = None

In [ ]:
path_processed_dataset = Path("output") / "processed_mitofish.parquet"
store = SequenceStore.from_parquet(path_processed_dataset, select_orders=select_orders, select_families=select_families,
                                   select_genera=select_genera, select_species=select_species,
                                   select_genes=select_genes)
store.summary()

In [ ]:
# ============================================================
# Load fixed hyperbolic tree embeddings and map transformer species to tree leaves
# ============================================================
import re
import torch
import torch.nn as nn

NODES_CSV = Path("output") / "processed_fishtree_nodes.csv"
HYPERBOLIC_CKPT_PATH = Path("output") / "hyperbolic_graph_embedding" / "node_embedding_model_hyperbolic.pt"
BALL_EPS = 1e-5


def project_to_poincare_ball(x: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    """Project points to the open unit Poincaré ball."""
    norm = torch.norm(x, dim=-1, keepdim=True).clamp_min(eps)
    max_norm = 1.0 - eps
    scale = torch.clamp(max_norm / norm, max=1.0)
    return x * scale


def poincare_distance(x: torch.Tensor, y: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    """Row-wise Poincaré distance. x and y must both have shape [B, D]."""
    x = x.float()
    y = y.float()
    x = project_to_poincare_ball(x, eps)
    y = project_to_poincare_ball(y, eps)

    x2 = torch.sum(x * x, dim=-1)
    y2 = torch.sum(y * y, dim=-1)
    diff2 = torch.sum((x - y) ** 2, dim=-1)

    denom = (1.0 - x2) * (1.0 - y2)
    z = 1.0 + 2.0 * diff2 / torch.clamp(denom, min=eps)
    return torch.acosh(torch.clamp(z, min=1.0 + eps))


def poincare_pairwise_distance(x: torch.Tensor, y: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    """Full pairwise Poincaré distance. x: [B, D], y: [N, D] -> [B, N]."""
    x = x.float()
    y = y.float()
    x = project_to_poincare_ball(x, eps)
    y = project_to_poincare_ball(y, eps)

    x2 = torch.sum(x * x, dim=-1, keepdim=True)       # [B, 1]
    y2 = torch.sum(y * y, dim=-1).unsqueeze(0)        # [1, N]
    diff2 = torch.cdist(x, y, p=2) ** 2               # [B, N]

    denom = (1.0 - x2) * (1.0 - y2)
    z = 1.0 + 2.0 * diff2 / torch.clamp(denom, min=eps)
    return torch.acosh(torch.clamp(z, min=1.0 + eps))


class HyperbolicNodeEmbeddingModel(nn.Module):
    """Same lightweight model class used by the graph embedding notebook."""

    def __init__(self, num_nodes: int, emb_dim: int = 256, init_scale: float = 1e-3, eps: float = 1e-5):
        super().__init__()
        self.embedding = nn.Embedding(num_nodes, emb_dim)
        self.eps = eps
        with torch.no_grad():
            self.embedding.weight.normal_(mean=0.0, std=init_scale)
            self.embedding.weight.copy_(project_to_poincare_ball(self.embedding.weight, eps=self.eps))

    def forward(self, node_ids: torch.Tensor) -> torch.Tensor:
        return project_to_poincare_ball(self.embedding(node_ids), eps=self.eps)

    def get_all_embeddings(self) -> torch.Tensor:
        return project_to_poincare_ball(self.embedding.weight, eps=self.eps)


def normalize_species_name(name: str) -> str:
    """Normalize species names from MitoFish and FishTree for robust exact matching."""
    name = str(name).strip().lower()
    name = name.replace("_", " ")
    name = re.sub(r"\s+", " ", name)
    return name


def infer_node_name_column(nodes_df: pd.DataFrame) -> str:
    """Try common node-name columns used in processed_fishtree_nodes.csv."""
    candidates = [
        "name", "node_name", "label", "taxon", "species", "scientific_name",
        "original_name", "tip_label", "leaf_name",
    ]
    for col in candidates:
        if col in nodes_df.columns:
            return col
    raise ValueError(
        "Could not infer the tree node name column. Available columns: "
        f"{list(nodes_df.columns)}. Set node_name_col manually below."
    )


def load_fixed_tree_embeddings(
    nodes_csv: str | Path,
    checkpoint_path: str | Path,
    store: SequenceStore,
    node_name_col: str | None = None,
    eps: float = 1e-5,
) -> tuple[torch.Tensor, torch.Tensor, pd.DataFrame, dict[int, int]]:
    """
    Returns
    -------
    leaf_prototypes_by_species : [num_valid_species, D]
        Fixed hyperbolic leaf embedding for each transformer species id.
    all_node_embeddings : [num_nodes, D]
        Fixed hyperbolic embeddings for leaves + internal nodes, used by profile loss.
    nodes_df : pd.DataFrame
        Tree node metadata.
    species_id_to_node_id : dict[int, int]
        Transformer species id -> tree node id.
    """
    nodes_df = pd.read_csv(nodes_csv)
    if node_name_col is None:
        node_name_col = infer_node_name_column(nodes_df)

    ckpt = torch.load(checkpoint_path, map_location="cpu")
    num_nodes = int(ckpt.get("num_nodes", len(nodes_df)))
    emb_dim = int(ckpt.get("emb_dim", 256))
    eps = float(ckpt.get("ball_eps", eps))

    node_model = HyperbolicNodeEmbeddingModel(num_nodes=num_nodes, emb_dim=emb_dim, eps=eps)
    node_model.load_state_dict(ckpt["model_state_dict"])
    node_model.eval()

    with torch.no_grad():
        all_node_embeddings = node_model.get_all_embeddings().detach().cpu().float()

    is_leaf_arr = nodes_df["is_leaf"].astype(bool).to_numpy() if "is_leaf" in nodes_df.columns else np.ones(len(nodes_df), dtype=bool)
    tree_name_to_node_id = {}
    duplicate_names = set()
    for node_id, row in nodes_df.iterrows():
        if not bool(is_leaf_arr[node_id]):
            continue
        key = normalize_species_name(row[node_name_col])
        if key in tree_name_to_node_id:
            duplicate_names.add(key)
        else:
            tree_name_to_node_id[key] = int(node_id)

    for key in duplicate_names:
        tree_name_to_node_id.pop(key, None)

    species_id_to_node_id = {}
    missing = []
    for species_id, species_name in enumerate(store.species_names):
        key = normalize_species_name(species_name)
        if key not in tree_name_to_node_id:
            missing.append(species_name)
            continue
        species_id_to_node_id[int(species_id)] = int(tree_name_to_node_id[key])

    if missing:
        print(f"Matched {len(species_id_to_node_id)} / {len(store.species_names)} transformer species to tree leaves.")
        print("First missing species:", missing[:20])
        raise ValueError(
            "Some transformer species were not found as unique tree leaves. "
            "Either adjust node_name_col/name normalization or filter select_species to matched species."
        )

    leaf_node_ids_by_species = torch.tensor(
        [species_id_to_node_id[i] for i in range(len(store.species_names))],
        dtype=torch.long,
    )
    leaf_prototypes_by_species = all_node_embeddings[leaf_node_ids_by_species].contiguous()

    print(f"Tree node name column: {node_name_col}")
    print("all_node_embeddings:", tuple(all_node_embeddings.shape))
    print("leaf_prototypes_by_species:", tuple(leaf_prototypes_by_species.shape))
    print("max all-node norm:", float(torch.norm(all_node_embeddings, dim=-1).max()))

    return leaf_prototypes_by_species, all_node_embeddings, nodes_df, species_id_to_node_id


leaf_prototypes_by_species, all_node_embeddings, nodes_df, species_id_to_node_id = load_fixed_tree_embeddings(
    nodes_csv=NODES_CSV,
    checkpoint_path=HYPERBOLIC_CKPT_PATH,
    store=store,
    node_name_col=None,  # set manually if automatic inference picks the wrong column
    eps=BALL_EPS,
)



In [ ]:
from dataclasses import dataclass
import numpy as np
import torch


@dataclass
class BatchBuilder:
    store: SequenceStore
    species_per_batch: int = 64
    crop_length: int = 512
    rng_seed: int | None = None

    def __post_init__(self) -> None:
        self.rng = np.random.default_rng(self.rng_seed)
        self.pad_id = self.store.token_to_id["PAD"]

        if self.species_per_batch < 1:
            raise ValueError("species_per_batch must be at least 1.")
        if self.crop_length < 1:
            raise ValueError("crop_length must be at least 1.")
        if self.species_per_batch > len(self.store.valid_species):
            raise ValueError(
                "species_per_batch cannot exceed the number of valid species."
            )

    @property
    def batch_size(self) -> int:
        return 2 * self.species_per_batch

    def sample_species(self) -> np.ndarray:
        return self.rng.choice(
            self.store.valid_species,
            size=self.species_per_batch,
            replace=False,
        )

    def sample_two_rows_for_species(self, species_id: int) -> tuple[tuple[int, int], tuple[int, int]]:
        gene_to_rows = self.store.species_to_gene_to_rows[species_id]
        gene_ids = list(gene_to_rows.keys())

        # Case A: species has >= 2 genes
        if len(gene_ids) >= 2:
            gene1, gene2 = self.rng.choice(gene_ids, size=2, replace=False)
            row1 = int(self.rng.choice(gene_to_rows[gene1]))
            row2 = int(self.rng.choice(gene_to_rows[gene2]))
            return (row1, gene1), (row2, gene2)

        # Only one gene available
        gene = gene_ids[0]
        rows = gene_to_rows[gene]

        # Case B: one gene, >= 2 rows
        if len(rows) >= 2:
            row1, row2 = self.rng.choice(rows, size=2, replace=False)
            return (int(row1), gene), (int(row2), gene)

        # Case C: one gene, one row
        row = rows[0]
        return (row, gene), (row, gene)

    def write_crop_into_numpy_arrays(self, seq: np.ndarray, input_ids: np.ndarray, attention_mask: np.ndarray,
                                     batch_idx: int) -> None:
        """
        Write one cropped/padded sequence into preallocated NumPy arrays.

        Parameters
        ----------
        seq
            1D uint8 encoded sequence
        input_ids
            Preallocated NumPy array of shape [B, L], dtype=int64
        attention_mask
            Preallocated NumPy array of shape [B, L], dtype=bool
        batch_idx
            Which row in the batch to fill
        """
        L = self.crop_length
        seq_len = len(seq)

        if seq_len >= L:
            if seq_len == L:
                crop = seq
            else:
                start = self.rng.integers(0, seq_len - L + 1)
                crop = seq[start:start + L]

            input_ids[batch_idx, :] = crop
            attention_mask[batch_idx, :] = True
            return

        # Shorter than crop length: pad
        input_ids[batch_idx, :] = self.pad_id
        attention_mask[batch_idx, :] = False

        input_ids[batch_idx, :seq_len] = seq
        attention_mask[batch_idx, :seq_len] = True

    def sample_batch_cpu(self) -> dict[str, torch.Tensor]:
        """
        Build one batch on CPU using NumPy first, then convert once to torch.
        """
        sampled_species = self.sample_species()
        B = self.batch_size
        L = self.crop_length

        input_ids_np = np.full((B, L), fill_value=self.pad_id, dtype=np.int64)
        attention_mask_np = np.zeros((B, L), dtype=bool)
        species_ids_np = np.empty(B, dtype=np.int64)
        gene_ids_np = np.empty(B, dtype=np.int64)

        batch_idx = 0
        for species_id in sampled_species:
            species_id = int(species_id)
            view1, view2 = self.sample_two_rows_for_species(species_id)

            for row_idx, gene_id in (view1, view2):
                seq = self.store.sequences[row_idx]
                self.write_crop_into_numpy_arrays(
                    seq=seq,
                    input_ids=input_ids_np,
                    attention_mask=attention_mask_np,
                    batch_idx=batch_idx,
                )
                species_ids_np[batch_idx] = species_id
                gene_ids_np[batch_idx] = int(gene_id)
                batch_idx += 1

        input_ids = torch.from_numpy(input_ids_np)
        attention_mask = torch.from_numpy(attention_mask_np)
        species_ids = torch.from_numpy(species_ids_np)
        gene_ids = torch.from_numpy(gene_ids_np)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "species_ids": species_ids,
            "gene_ids": gene_ids,
        }

In [ ]:
builder = BatchBuilder(
    store=store,
    species_per_batch=64,
    crop_length=512,
    rng_seed=None,
)

batch = builder.sample_batch_cpu()

for i in range(0, len(batch["species_ids"]), 2):
    sid1 = int(batch["species_ids"][i].item())
    sid2 = int(batch["species_ids"][i + 1].item())
    gid1 = int(batch["gene_ids"][i].item())
    gid2 = int(batch["gene_ids"][i + 1].item())

    print(
        f"pair {i//2}: species={store.species_names[sid1]}, "
        f"genes=({store.gene_names[gid1]}, {store.gene_names[gid2]})"
    )

In [ ]:
from torch.utils.data import IterableDataset, DataLoader, get_worker_info


class ContrastiveBatchDataset(IterableDataset):
    """
    IterableDataset that keeps your existing BatchBuilder logic and lets
    DataLoader workers prebuild full batches in the background.
    """

    def __init__(self, store: SequenceStore, species_per_batch: int = 64, crop_length: int = 512,
                 base_seed: int | None = None) -> None:
        super().__init__()
        self.store = store
        self.species_per_batch = species_per_batch
        self.crop_length = crop_length
        self.base_seed = base_seed

    def __iter__(self):
        worker = get_worker_info()

        if worker is None:
            worker_id = 0
        else:
            worker_id = worker.id

        rng_seed = self.base_seed if self.base_seed is None else self.base_seed + worker_id
        # Each worker gets its own RNG seed, so they do not build identical batches
        builder = BatchBuilder(
            store=self.store,
            species_per_batch=self.species_per_batch,
            crop_length=self.crop_length,
            rng_seed=rng_seed,
        )

        while True:
            yield builder.sample_batch_cpu()


def make_batch_loader(store: SequenceStore, species_per_batch: int = 64, crop_length: int = 512,
                      num_workers: int = 4, pin_memory: bool = True, prefetch_factor: int = 2,
                      persistent_workers: bool = True, base_seed: int | None = None) -> DataLoader:
    dataset = ContrastiveBatchDataset(
        store=store,
        species_per_batch=species_per_batch,
        crop_length=crop_length,
        base_seed=base_seed,
    )

    loader_kwargs = dict(
        dataset=dataset,
        batch_size=None,   # dataset already yields complete batches
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    if num_workers > 0:
        loader_kwargs["prefetch_factor"] = prefetch_factor
        loader_kwargs["persistent_workers"] = persistent_workers

    return DataLoader(**loader_kwargs)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class SequenceEncoder(nn.Module):
    """
    Transformer-based encoder for DNA sequences with optional gene embeddings.

    Input:
        input_ids:      [B, L] torch.long
        attention_mask: [B, L] torch.bool
            True  = real token
            False = padding
        gene_ids:       [B] torch.long
            gene identity for each sequence

    Output:
        z_seq: [B, embed_dim]
            projected into the open Poincaré ball
    """

    def __init__(self, vocab_size: int, max_length: int, num_genes: int, d_model: int = 256, nhead: int = 8,
                 num_layers: int = 4, dim_feedforward: int = 1024, dropout: float = 0.1, embed_dim: int = 256,
                 pad_token_id: int = 0, use_gene_embedding: bool = True,
                 poincare_eps: float = 1e-5, sequence_output_scale: float = 0.05) -> None:
        super().__init__()

        if d_model % nhead != 0:
            raise ValueError("d_model must be divisible by nhead.")

        self.vocab_size = vocab_size
        self.max_length = max_length
        self.num_genes = num_genes
        self.d_model = d_model
        self.embed_dim = embed_dim
        self.pad_token_id = pad_token_id
        self.use_gene_embedding = use_gene_embedding
        self.poincare_eps = poincare_eps
        self.sequence_output_scale = sequence_output_scale

        # Token embedding
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=d_model,
            padding_idx=pad_token_id,
        )

        # Positional embedding
        self.position_embedding = nn.Embedding(
            num_embeddings=max_length,
            embedding_dim=d_model,
        )

        # Optional gene embedding
        if self.use_gene_embedding:
            self.gene_embedding = nn.Embedding(
                num_embeddings=num_genes,
                embedding_dim=d_model,
            )

        # Input dropout
        self.input_dropout = nn.Dropout(dropout)

        # Transformer encoder stack
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_layers,
        )

        # Projection head
        self.projection = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, embed_dim),
        )

    def masked_mean_pool(self, x: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """
        x: [B, L, d_model]
        attention_mask: [B, L] bool, True=real token, False=padding
        """
        mask = attention_mask.unsqueeze(-1).to(dtype=x.dtype)  # [B, L, 1]
        x_masked = x * mask
        summed = x_masked.sum(dim=1)  # [B, d_model]
        counts = mask.sum(dim=1).clamp(min=1.0)  # [B, 1]
        return summed / counts

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                gene_ids: torch.Tensor | None = None) -> torch.Tensor:
        """
        input_ids:      [B, L] torch.long
        attention_mask: [B, L] torch.bool
        gene_ids:       [B] torch.long or None

        returns:
            z_seq: [B, embed_dim], projected into the open Poincaré ball
        """
        if input_ids.ndim != 2:
            raise ValueError(f"input_ids must have shape [B, L], got {input_ids.shape}")
        if attention_mask.ndim != 2:
            raise ValueError(
                f"attention_mask must have shape [B, L], got {attention_mask.shape}"
            )
        if input_ids.shape != attention_mask.shape:
            raise ValueError(
                f"input_ids and attention_mask must have same shape, got "
                f"{input_ids.shape} and {attention_mask.shape}"
            )

        batch_size, seq_len = input_ids.shape

        if seq_len > self.max_length:
            raise ValueError(
                f"Sequence length {seq_len} exceeds max_length={self.max_length}"
            )

        if self.use_gene_embedding:
            if gene_ids is None:
                raise ValueError("gene_ids must be provided when use_gene_embedding=True")
            if gene_ids.ndim != 1 or gene_ids.shape[0] != batch_size:
                raise ValueError(
                    f"gene_ids must have shape [B], got {gene_ids.shape}"
                )

        # Position indices: [B, L]
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(
            batch_size, seq_len
        )

        # Token + position embeddings
        x = self.token_embedding(input_ids) + self.position_embedding(positions)

        # Add gene embedding if enabled
        if self.use_gene_embedding:
            gene_vec = self.gene_embedding(gene_ids)          # [B, d_model]
            gene_vec = gene_vec.unsqueeze(1)                  # [B, 1, d_model]
            x = x + gene_vec                                  # broadcast over L

        # Input dropout
        x = self.input_dropout(x)

        # PyTorch transformer expects True for positions to ignore
        src_key_padding_mask = ~attention_mask  # [B, L]

        x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)  # [B, L, d_model]

        pooled = self.masked_mean_pool(x, attention_mask)    # [B, d_model]
        z_seq = self.projection(pooled)                      # [B, embed_dim]

        # Important change from the Euclidean notebook:
        # do not L2-normalize here. Unit-norm vectors lie on the Poincaré
        # boundary. Instead, start sequence embeddings safely near the origin
        # and let training move them toward their fixed leaf prototypes.
        z_seq = project_to_poincare_ball(
            self.sequence_output_scale * z_seq,
            eps=self.poincare_eps,
        )

        return z_seq


In [ ]:
class SpeciesContrastiveModel(nn.Module):
    """
    Sequence encoder aligned to fixed hyperbolic tree embeddings.

    Training objective:
        - batch-restricted fixed-leaf prototype loss using -Poincaré distance logits
        - half-vs-half sequence contrastive loss using -Poincaré distance logits
        - sampled profile loss: sequence-to-tree-node distance profile should match
          the true species leaf-to-tree-node distance profile
    """

    def __init__(
        self,
        encoder: SequenceEncoder,
        leaf_prototypes_by_species: torch.Tensor,
        all_node_embeddings: torch.Tensor,
        temperature: float = 0.07,
        lambda_proto: float = 1.0,
        lambda_seq: float = 0.25,
        lambda_profile: float = 0.1,
        profile_num_nodes: int = 256,
        poincare_eps: float = 1e-5,
    ) -> None:
        super().__init__()

        if temperature <= 0:
            raise ValueError("temperature must be > 0")
        if leaf_prototypes_by_species.ndim != 2:
            raise ValueError("leaf_prototypes_by_species must have shape [num_species, D]")
        if all_node_embeddings.ndim != 2:
            raise ValueError("all_node_embeddings must have shape [num_nodes, D]")
        if leaf_prototypes_by_species.shape[1] != all_node_embeddings.shape[1]:
            raise ValueError("leaf and all-node embeddings must have the same embedding dimension")

        self.encoder = encoder
        self.temperature = temperature
        self.lambda_proto = lambda_proto
        self.lambda_seq = lambda_seq
        self.lambda_profile = lambda_profile
        self.profile_num_nodes = profile_num_nodes
        self.poincare_eps = poincare_eps

        # Fixed tree embeddings. Buffers move with .to(device), but are not optimized.
        self.register_buffer(
            "leaf_prototypes_by_species",
            project_to_poincare_ball(leaf_prototypes_by_species.float(), eps=poincare_eps),
            persistent=True,
        )
        self.register_buffer(
            "all_node_embeddings",
            project_to_poincare_ball(all_node_embeddings.float(), eps=poincare_eps),
            persistent=True,
        )

    def sample_profile_nodes(self, device: torch.device) -> torch.Tensor:
        num_nodes = self.all_node_embeddings.shape[0]
        k = min(int(self.profile_num_nodes), num_nodes)
        return torch.randperm(num_nodes, device=device)[:k]

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        species_ids: torch.Tensor,
        gene_ids: torch.Tensor,
    ) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        """
        Assumes the batch is organized in paired views:
            0,1   = same species
            2,3   = same species
            4,5   = same species
            ...
        """
        z_seq = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            gene_ids=gene_ids,
        )  # [B, D], already projected into Poincaré ball

        B = z_seq.shape[0]
        device = z_seq.device

        if B % 2 != 0:
            raise ValueError(f"Batch size must be even, got {B}")

        # ------------------------------------------------------------
        # 1) Batch-restricted fixed leaf prototype loss
        # ------------------------------------------------------------
        batch_species_ids, local_targets = torch.unique(
            species_ids,
            sorted=True,
            return_inverse=True,
        )  # batch_species_ids: [S], local_targets: [B]

        z_species_batch = self.leaf_prototypes_by_species[batch_species_ids]  # [S, D]
        d_proto = poincare_pairwise_distance(z_seq, z_species_batch, eps=self.poincare_eps)  # [B, S]
        logits_proto = -d_proto / self.temperature
        loss_proto = F.cross_entropy(logits_proto, local_targets)

        # ------------------------------------------------------------
        # 2) Half-vs-half CLIP-style sequence contrastive loss
        # ------------------------------------------------------------
        z_a = z_seq[0::2]   # [S, D]
        z_b = z_seq[1::2]   # [S, D]

        S = z_a.shape[0]
        targets_seq = torch.arange(S, device=device)

        d_seq = poincare_pairwise_distance(z_a, z_b, eps=self.poincare_eps)  # [S, S]
        logits_seq = -d_seq / self.temperature

        loss_seq_ab = F.cross_entropy(logits_seq, targets_seq)
        loss_seq_ba = F.cross_entropy(logits_seq.T, targets_seq)
        loss_seq = 0.5 * (loss_seq_ab + loss_seq_ba)

        # ------------------------------------------------------------
        # 3) Sampled distance-profile loss against all tree nodes
        # ------------------------------------------------------------
        if self.lambda_profile > 0.0 and self.profile_num_nodes > 0:
            profile_node_ids = self.sample_profile_nodes(device=device)
            z_profile_nodes = self.all_node_embeddings[profile_node_ids]       # [K, D]
            z_true_leaf = self.leaf_prototypes_by_species[species_ids]         # [B, D]

            d_seq_nodes = poincare_pairwise_distance(z_seq, z_profile_nodes, eps=self.poincare_eps)
            with torch.no_grad():
                d_leaf_nodes = poincare_pairwise_distance(z_true_leaf, z_profile_nodes, eps=self.poincare_eps)

            loss_profile = F.smooth_l1_loss(d_seq_nodes, d_leaf_nodes)
        else:
            profile_node_ids = torch.empty(0, dtype=torch.long, device=device)
            loss_profile = z_seq.new_tensor(0.0)

        # ------------------------------------------------------------
        # 4) Combine losses
        # ------------------------------------------------------------
        total_loss = (
            self.lambda_proto * loss_proto
            + self.lambda_seq * loss_seq
            + self.lambda_profile * loss_profile
        )

        outputs = {
            "z_seq": z_seq,
            "batch_species_ids": batch_species_ids,
            "z_species_batch": z_species_batch,
            "logits_proto": logits_proto,
            "logits_seq": logits_seq,
            "loss_proto": loss_proto.detach(),
            "loss_seq": loss_seq.detach(),
            "loss_profile": loss_profile.detach(),
            "local_targets": local_targets,
            "targets_seq": targets_seq,
            "profile_node_ids": profile_node_ids,
        }
        return total_loss, outputs


In [ ]:
import time
from pathlib import Path

import torch


def topk_accuracy(logits: torch.Tensor, targets: torch.Tensor, k: int = 1) -> torch.Tensor:
    """
    Compute top-k accuracy.

    logits:  [N, C]
    targets: [N]
    """
    k = min(k, logits.shape[1])
    topk = logits.topk(k, dim=1).indices
    correct = (topk == targets.unsqueeze(1)).any(dim=1)
    return correct.float().mean()


def save_checkpoint(save_path: str | Path, model: SpeciesContrastiveModel, optimizer: torch.optim.Optimizer,
                    step: int, loss: float, loss_proto: float, loss_seq: float, loss_profile: float,
                    acc_proto_top1: float, acc_proto_top5: float, acc_seq: float,
                    species_names: list[str]) -> None:
    """
    Save a training checkpoint including model weights and the current
    fixed tree embeddings and metadata for downstream analysis.
    """
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    checkpoint = {
        "step": step,
        "loss": loss,
        "loss_proto": loss_proto,
        "loss_seq": loss_seq,
        "loss_profile": loss_profile,
        "acc_proto_top1": acc_proto_top1,
        "acc_proto_top5": acc_proto_top5,
        "acc_seq": acc_seq,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "fixed_leaf_prototypes_by_species": model.leaf_prototypes_by_species.detach().cpu(),
        "fixed_all_node_embeddings": model.all_node_embeddings.detach().cpu(),
        "species_names": species_names,
    }

    torch.save(checkpoint, save_path)

def move_batch_to_device(batch: dict[str, torch.Tensor], device: str | torch.device,
                         non_blocking: bool = True) -> dict[str, torch.Tensor]:
    device = torch.device(device)
    return {
        k: v.to(device, non_blocking=non_blocking)
        for k, v in batch.items()
    }

def train_species_contrastive(model: SpeciesContrastiveModel, loader: DataLoader, optimizer: torch.optim.Optimizer,
                              device: str | torch.device, num_steps: int, checkpoint_dir: str | Path,
                              checkpoint_every: int = 500, log_every: int = 50, grad_clip_norm: float | None = 1.0,
                              start_step: int = 0, use_amp: bool = True,
                              species_names: list[str] | None = None) -> list[dict]:
    """
    Train the species-contrastive model with optional AMP.

    Periodically saves checkpoints containing:
    - encoder weights
    - optimizer state
    - fixed tree embedding buffers
    - metadata (step, losses, accuracies)

    Returns
    -------
    history : list of dict
        Per-log-step training metrics.
    """
    device = torch.device(device)
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    amp_enabled = use_amp and device.type == "cuda"
    scaler = torch.GradScaler("cuda", enabled=amp_enabled)

    model.train()
    history = []

    t0 = time.time()
    batch_iter = iter(loader)

    for step in range(start_step + 1, start_step + num_steps + 1):
        # Build batch on CPU, then move once to GPU
        batch_cpu = next(batch_iter)
        batch = move_batch_to_device(batch_cpu, device=device, non_blocking=True)

        with torch.autocast("cuda", enabled=amp_enabled):
            loss, outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                species_ids=batch["species_ids"],
                gene_ids=batch["gene_ids"],
            )

        logits_proto = outputs["logits_proto"]   # [B, S]
        logits_seq = outputs["logits_seq"]       # [S, S]
        local_targets = outputs["local_targets"] # [B]
        targets_seq = outputs["targets_seq"]     # [S]

        # Metrics are computed outside autocast for consistency/readability
        acc_proto_top1 = topk_accuracy(logits_proto.float(), local_targets, k=1)
        acc_proto_top5 = topk_accuracy(logits_proto.float(), local_targets, k=5)
        acc_seq = topk_accuracy(logits_seq.float(), targets_seq, k=1)

        loss_proto = outputs["loss_proto"]
        loss_seq = outputs["loss_seq"]
        loss_profile = outputs["loss_profile"]

        z_seq = outputs["z_seq"]
        z_true_leaf = model.leaf_prototypes_by_species[batch["species_ids"]]

        seq_radii = z_seq.norm(dim=-1)
        leaf_radii = z_true_leaf.norm(dim=-1)

        mean_seq_radius = seq_radii.mean()
        mean_leaf_radius = leaf_radii.mean()
        max_seq_radius = seq_radii.max()
        max_leaf_radius = leaf_radii.max()

        d_true_leaf = poincare_distance(z_seq, z_true_leaf, eps=model.poincare_eps)
        mean_d_true_leaf = d_true_leaf.mean()

        optimizer.zero_grad(set_to_none=True)

        if amp_enabled:
            scaler.scale(loss).backward()

            if grad_clip_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()

            if grad_clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

            optimizer.step()

        # Logging
        if step % log_every == 0 or step == start_step + 1:
            elapsed = time.time() - t0
            metrics = {
                "step": step,
                "loss": float(loss.item()),
                "loss_proto": float(loss_proto.item()),
                "loss_seq": float(loss_seq.item()),
                "loss_profile": float(loss_profile.item()),
                "acc_proto_top1": float(acc_proto_top1.item()),
                "acc_proto_top5": float(acc_proto_top5.item()),
                "acc_seq": float(acc_seq.item()),
                "mean_seq_radius": float(mean_seq_radius.item()),
                "mean_leaf_radius": float(mean_leaf_radius.item()),
                "max_seq_radius": float(max_seq_radius.item()),
                "max_leaf_radius": float(max_leaf_radius.item()),
                "mean_d_true_leaf": float(mean_d_true_leaf.item()),
                "elapsed_sec": elapsed,
            }
            history.append(metrics)

            print(
                f"[step {step:6d}] "
                f"loss={metrics['loss']:.4f} "
                f"proto={metrics['loss_proto']:.4f} "
                f"seq={metrics['loss_seq']:.4f} "
                f"profile={metrics['loss_profile']:.4f} "
                f"acc1={metrics['acc_proto_top1']:.4f} "
                f"acc5={metrics['acc_proto_top5']:.4f} "
                f"acc_seq={metrics['acc_seq']:.4f} "
                f"r_seq={metrics['mean_seq_radius']:.4f}/{metrics['max_seq_radius']:.4f} "
                f"r_leaf={metrics['mean_leaf_radius']:.4f}/{metrics['max_leaf_radius']:.4f} "
                f"d_leaf={metrics['mean_d_true_leaf']:.4f} "
                f"time={elapsed:.1f}s"
            )
            
        # Periodic checkpointing
        if step % checkpoint_every == 0 or step == start_step + num_steps:
            save_path = checkpoint_dir / f"step_{step:07d}.pt"
            save_checkpoint(
                save_path=save_path,
                model=model,
                optimizer=optimizer,
                step=step,
                loss=float(loss.item()),
                loss_proto=float(loss_proto.item()),
                loss_seq=float(loss_seq.item()),
                loss_profile=float(loss_profile.item()),
                acc_proto_top1=float(acc_proto_top1.item()),
                acc_proto_top5=float(acc_proto_top5.item()),
                acc_seq=float(acc_seq.item()),
                species_names=species_names if species_names is not None else [],
            )
            print(f"Saved checkpoint to {save_path}")

    return history


In [ ]:
checkpoint_dir: str = "output/checkpoints_transformer_hyperbolic"
resume_ckpt_path: str | None = None
num_steps: int = 500_000
species_per_batch: int = 64
crop_length: int = 512

# Hyperbolic contrastive settings
temperature: float = 0.07
lambda_proto: float = 1.0
lambda_seq: float = 0.25
lambda_profile: float = 0.1
profile_num_nodes: int = 256
sequence_output_scale: float = 0.05


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loader = make_batch_loader(
    store=store,
    species_per_batch=species_per_batch,
    crop_length=crop_length,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True,
    base_seed=None
)

encoder = SequenceEncoder(
    vocab_size=store.vocab_size,
    max_length=crop_length,
    num_genes=store.num_genes,
    d_model=256,
    nhead=8,
    num_layers=4,
    dim_feedforward=1024,
    dropout=0.1,
    embed_dim=256,
    pad_token_id=store.token_to_id["PAD"],
    use_gene_embedding=True,
    poincare_eps=BALL_EPS,
    sequence_output_scale=sequence_output_scale,
).to(device)

model = SpeciesContrastiveModel(
    encoder=encoder,
    leaf_prototypes_by_species=leaf_prototypes_by_species,
    all_node_embeddings=all_node_embeddings,
    temperature=temperature,
    lambda_proto=lambda_proto,
    lambda_seq=lambda_seq,
    lambda_profile=lambda_profile,
    profile_num_nodes=profile_num_nodes,
    poincare_eps=BALL_EPS,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-2,
)

start_step = 0
if resume_ckpt_path is not None:
    checkpoint = torch.load(resume_ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_step = checkpoint["step"]

print(f"Resuming from step {start_step}")

history = train_species_contrastive(
    model=model,
    loader=loader,
    optimizer=optimizer,
    device=device,
    num_steps=num_steps,
    checkpoint_dir=checkpoint_dir,
    checkpoint_every=10_000,
    log_every=100,
    grad_clip_norm=1.0,
    start_step=start_step,
    use_amp=True,
    species_names=store.species_names
)
